In [ ]:
import os

import sys
sys.path.append("../src")

from juris_summarizer import get_client, chunk_text, summarize_chunk, run_batch_eval
from juris_summarizer.prompts import format_chunk_summaries

In [2]:
map_client = get_client("groq")
reduce_client = get_client("cerebras")

In [3]:
import pandas as pd 
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

df = pd.read_csv("../data/train.csv")
df['text_words'] = df['text'].apply(lambda x: len(str(x).split()))
df['summary_words'] = df['summary'].str.split().apply(len)
df['compression_ratio'] = df['summary_words'] / df['text_words']
df['text_tokens'] = df['text'].apply(lambda x: len(enc.encode(x)))

In [4]:
df['cr_decile'] = pd.qcut(df['compression_ratio'], 10, labels=False)
sample = df.groupby('cr_decile', group_keys=False).apply(lambda g: g.sample(1, random_state=42))

In [5]:
# few-shot example — map step uses map_client
example_idx = 713
example_reference_summary = df.loc[example_idx, "summary"]
example_chunks = chunk_text(df.loc[example_idx, "text"])

example_summaries = [
    summarize_chunk(map_client, c, i + 1, len(example_chunks))
    for i, c in enumerate(example_chunks)
]

example_chunk_summaries = format_chunk_summaries(example_summaries)

# batch eval — pass both clients through
results_df = run_batch_eval(
    map_client, reduce_client, sample, example_chunk_summaries, example_reference_summary
)

100%|██████████| 10/10 [12:01<00:00, 72.17s/it]


In [6]:
results_df.describe()

,paper_id,text_words,compression_ratio,generated_words,reference_words,rouge2_precision,rouge2_recall,rouge2_fmeasure
count,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000
mean,829.000000,5571.200000,0.034588,271.200000,173.500000,0.056400,0.097506,0.067566
std,29.884965,2150.067999,0.016344,54.879261,66.431669,0.022523,0.037964,0.018633
min,768.000000,2241.000000,0.015252,191.000000,70.000000,0.031785,0.052326,0.044346
25%,822.750000,4865.500000,0.022440,233.000000,147.750000,0.045197,0.075814,0.056081
50%,834.500000,5465.000000,0.029299,265.500000,158.000000,0.052567,0.089099,0.066501
75%,846.500000,6211.000000,0.045764,296.500000,203.500000,0.054970,0.109551,0.074022
max,867.000000,9835.000000,0.063235,386.000000,301.000000,0.108974,0.178082,0.109501


In [7]:
import os
print(os.path.exists("results_checkpoint.jsonl"))
if os.path.exists("results_checkpoint.jsonl"):
    with open("results_checkpoint.jsonl") as f:
        print(sum(1 for _ in f), "lines")

True
10 lines


In [8]:
results_df = pd.read_json("results_checkpoint.jsonl", lines=True)
for i, row in results_df.iterrows():
    print(row['paper_id'], row['generated_words'], row['generated'][:100])

790 236 This paper investigates the determinants of city‑county consolidation referenda, integrating a revie
835 312 This paper extends Schelling’s classic segregation framework by embedding agents in explicit social‑
820 295 The rapid expansion of quantitative research on forced displacement has been enabled by unprecedente
834 232 This study investigates the role of Japanese popular culture (JPC) in shaping Australian learners’ m
855 255 This paper traces the emergence of the modern Japanese “girl” through the formative role of early gi
867 386 Colombia has reduced violence and expanded social welfare after decades of armed conflict, yet infor
768 276 This study investigates Turkish EFL students’ beliefs and expectations regarding university‑level tr
842 232 Second‑language acquisition research consistently documents affective and cognitive obstacles for le
831 191 This paper reconceptualises augmented reality (AR) from an ocular‑centric overlay to a fully multise
848 297 This study 

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=os.environ.get("HF_TOKEN"))
response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {
            "role": "user", 
            "content": "Why is fast inference important?",
        }
    ],
)

print(response.choices[0].message.content)